In [1]:
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder

import torch 
from torchvision.datasets import ImageFolder

#veri setleriinin yollarını tanımlıyoruz.

nwpu_train_path = "../split_data/nwpu/train"
nwpu_val_path = "../split_data/nwpu/val"
nwpu_test_path = "../split_data/nwpu/test"

ucmerced_train_path = "../split_data/ucmerced/train"
ucmerced_val_path = "../split_data/ucmerced/val"
ucmerced_test_path = "../split_data/ucmerced/test"
###############
#NWPU veri seti ile ilgili bilgiler


#nwpu train verisi için kullanılacak transform işlemleri
from torchvision import transforms

nwpu_train_tranform= transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(p=0.3),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],[0.229, 0.224, 0.225])


])

#nwpu test ve validasyon verisi için kullanılacak transform işlemleri
nwpu_test_tranform= transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],[0.229, 0.224, 0.225])


])

#ucmerced train verisi için kullanılacak transform işlemleri

ucmerced_train_tranform= transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(p=0.3),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],[0.229, 0.224, 0.225])


])

#ucmerced test,validasyon verisi için kullanılacak transform işlemleri

ucmerced_test_tranform= transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],[0.229, 0.224, 0.225])


])

nwpu_train = ImageFolder(nwpu_train_path,transform=nwpu_train_tranform)
nwpu_val = ImageFolder(nwpu_val_path,transform=nwpu_test_tranform)
nwpu_test = ImageFolder(nwpu_test_path,transform=nwpu_test_tranform)
nwpu_toplam_resim_sayisi = len(nwpu_val)+ len(nwpu_train) +len(nwpu_test)
nwpu_siniflar = nwpu_train.classes

#Image Folder (görsel,sınıf) şeklinde tuple verir

ucmerced_train = ImageFolder(ucmerced_train_path,transform=ucmerced_train_tranform)
ucmerced_val = ImageFolder(ucmerced_val_path,transform=ucmerced_test_tranform)
ucmerced_test = ImageFolder(ucmerced_test_path,transform=ucmerced_test_tranform)
ucmerced_toplam_resim_sayisi = len(ucmerced_val)+ len(ucmerced_train) +len(ucmerced_test)
ucmerced_siniflar = ucmerced_train.classes

#nwpu verisetindeki train,validasyon ve test verileri için DataLoder objeleri oluşturuldu

nwpu_train_loader=DataLoader(dataset=nwpu_train,batch_size=16,shuffle=True)
nwpu_valid_loader=DataLoader(dataset=nwpu_val,batch_size=16,shuffle=False)
nwpu_test_loader=DataLoader(dataset=nwpu_test,batch_size=16,shuffle=False)

#ucmerced verisetindeki train,validasyon ve test verileri için DataLoder objeleri oluşturuldu

ucmerced_train_loader = DataLoader(dataset=ucmerced_train,batch_size=16,shuffle=True)
ucmerced_valid_loader = DataLoader(dataset=ucmerced_val,batch_size=16,shuffle=False)
ucmerced_test_loader = DataLoader(dataset=ucmerced_test,batch_size=16,shuffle=True)




##GPU kontrolü

In [2]:
import torch


#nvdia GPU sahip cihazlar için kontrol
#device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#print(device)

#mac cihazlar için GPU kontrolü
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(device)

mps


##Sıfırdan kendi modelimizi tanımlıcaz.TinyVGG benzeri bir model olacak ama biraz daha derin

In [3]:
from torch import nn
import torch

class kendiModelim(nn.Module):
    def __init__(self,giris_boyutu:int,ara_boyut:int,cikis_boyutu:int):
        super().__init__()

        self.blok1 = nn.Sequential(
            nn.Conv2d(
            in_channels=giris_boyutu,
            out_channels=ara_boyut,
            kernel_size=3,
            padding=1,
            stride=1,
            ),
            nn.ReLU(),
            nn.Conv2d(
                in_channels=ara_boyut,
                out_channels=ara_boyut,
                kernel_size=3,
                padding=1,
                stride=1
            ),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2,stride=2)

        )

        self.blok2 = nn.Sequential(
            nn.Conv2d(in_channels=ara_boyut,
                      out_channels=ara_boyut*2,
                      kernel_size=3,
                      padding=1,
                      stride=1),
            nn.ReLU(),
            nn.Conv2d(in_channels=ara_boyut*2,
                      out_channels=ara_boyut*2,
                      kernel_size=3,
                      stride=1,
                      padding=1),
            nn.ReLU(),
            nn.MaxPool2d(stride=2,kernel_size=2)
        )

        self.siniflandirici = nn.Sequential(
            nn.AdaptiveAvgPool2d((1,1)),
            nn.Flatten(),
            nn.Dropout(p=0.1),
            nn.Linear(in_features=ara_boyut*2,out_features=cikis_boyutu)


        )

    def forward(self,X:torch.Tensor):
        return self.siniflandirici(self.blok2(self.blok1(X)))

torch.manual_seed(42)
kendiModelim_nwpu = kendiModelim(giris_boyutu=3,ara_boyut=32,cikis_boyutu=len(nwpu_train.classes))
kendiModelim_ucmerced = kendiModelim(giris_boyutu=3,ara_boyut=32,cikis_boyutu=len(ucmerced_train.classes))



In [4]:
from torchinfo import summary


summary(model=kendiModelim_nwpu,input_size=(1,3,224,224))

Layer (type:depth-idx)                   Output Shape              Param #
kendiModelim                             [1, 45]                   --
├─Sequential: 1-1                        [1, 32, 112, 112]         --
│    └─Conv2d: 2-1                       [1, 32, 224, 224]         896
│    └─ReLU: 2-2                         [1, 32, 224, 224]         --
│    └─Conv2d: 2-3                       [1, 32, 224, 224]         9,248
│    └─ReLU: 2-4                         [1, 32, 224, 224]         --
│    └─MaxPool2d: 2-5                    [1, 32, 112, 112]         --
├─Sequential: 1-2                        [1, 64, 56, 56]           --
│    └─Conv2d: 2-6                       [1, 64, 112, 112]         18,496
│    └─ReLU: 2-7                         [1, 64, 112, 112]         --
│    └─Conv2d: 2-8                       [1, 64, 112, 112]         36,928
│    └─ReLU: 2-9                         [1, 64, 112, 112]         --
│    └─MaxPool2d: 2-10                   [1, 64, 56, 56]           --
├─S

In [2]:
import torchvision

efficientnet_agirliklar = torchvision.models.EfficientNet_B0_Weights.DEFAULT
efficientnet_model = torchvision.models.efficientnet_b0(weights=efficientnet_agirliklar)




Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /Users/muhammedcakici/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100.0%


In [ ]:
import torchinfo
